In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("catalog_name", "dbr_dev_ua5816bd", "Catalog Name")
dbutils.widgets.text("bronze_schema", "oles0305_bronze", "Bronze Schema")
dbutils.widgets.text("silver_schema", "oles0305_silver", "Silver Schema")

CATALOG_NAME = dbutils.widgets.get("catalog_name")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")

BRONZE = f"{CATALOG_NAME}.{BRONZE_SCHEMA}"
SILVER = f"{CATALOG_NAME}.{SILVER_SCHEMA}"

In [0]:
df_bronze = spark.read.table(f"{BRONZE}.results")

df_clean = (
    df_bronze
        .filter(F.col("_rescued_data").isNull() & F.col("points").isNotNull())
        .select(
            F.col("resultId").cast("int"),
            F.col("raceId").cast("int"),
            F.col("driverId").cast("int"),
            F.col("constructorId").cast("int"),
            F.col("grid").cast("int"),
            F.col("position").cast("int"),
            F.col("points").cast("double"),
            F.col("laps").cast("int"),
            F.col("milliseconds").cast("int"),
            F.col("statusId").cast("int"),
            F.col("time").cast("string"),
            F.col("ingested_at").alias("bronze_ingested_at"),
            F.current_timestamp().alias("silver_processed_at")
        )
        .dropDuplicates(["resultId"])
)

In [0]:
silver_table = DeltaTable.forName(spark, f"{SILVER}.results")

In [0]:
(   # SCD Type 1
    silver_table.alias("target")
        .merge(
            df_clean.alias("source"), 
            "target.resultId = source.resultId"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
)